# 03. 미들웨어 기반 컨텍스트 엔지니어링 (Context Engineering with Middleware)

> 이 노트북은 LangChain v1 **미들웨어**(`@dynamic_prompt`, `@wrap_model_call`, `SummarizationMiddleware`, `Store`)를 중심으로 컨텍스트를 엔지니어링해요. Deep Agents SDK를 기반으로 한 5가지 컨텍스트 타입 관점은 **`10_Deep_Agents/03-Context-Engineering.ipynb`** 에서 따로 다뤄요.

에이전트를 구축할 때 가장 어려운 부분은 **신뢰성(reliability)**을 확보하는 것이에요. MVP 단계에서 잘 동작하던 에이전트가 실제 환경에서 실패하는 이유는 대부분 하나예요 -- LLM에 **올바른 컨텍스트**가 전달되지 않아서입니다.

**컨텍스트 엔지니어링(Context Engineering)**은 LLM이 작업을 성공적으로 완수할 수 있도록 올바른 정보, 도구, 지시사항을 올바른 형식으로 제공하는 기술이에요. 이것이 AI 엔지니어의 가장 중요한 역량 중 하나입니다.

> 🔑 **핵심 개념**: 프롬프트 엔지니어링이 "LLM에게 **무엇을** 시킬까?"라면, 컨텍스트 엔지니어링은 "LLM이 **성공하려면** 어떤 정보가 필요할까?"예요. 비유하자면, 프롬프트 엔지니어링은 시험 문제를 잘 내는 것이고, 컨텍스트 엔지니어링은 학생에게 **필요한 참고 자료를 적절한 시점에 제공**하는 것이에요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **컨텍스트 3종류**(Model, Tool, Life-cycle)의 차이와 지속성을 설명할 수 있어요
2. **데이터 소스 3가지**(Runtime Context, State, Store)를 적절히 선택해 활용할 수 있어요
3. `@dynamic_prompt`로 State/Store/Runtime Context 기반 **동적 시스템 프롬프트**를 구현할 수 있어요
4. `@wrap_model_call`로 도구 필터링과 모델 선택을 **런타임에 동적으로 제어**할 수 있어요
5. **RBAC(역할 기반 접근 제어)**를 미들웨어로 구현할 수 있어요

## 사전 지식

- `02-Human-In-The-Loop-V1.ipynb`: 미들웨어 기반 HITL, HumanInTheLoopMiddleware
- `01-Middleware-Basics.ipynb`: `@dynamic_prompt`, `@wrap_model_call` 기본 사용법

## 컨텍스트 엔지니어링 전체 구조

이 장의 핵심은 **"무엇을 어디서 읽고, 무엇을 바꾸는가?"**를 분리해서 보는 것입니다. 컨텍스트 엔지니어링은 아래 흐름으로 동작해요.

> **데이터 소스에서 값을 읽고 → 미들웨어/도구가 결정을 내리고 → 모델 호출·도구 실행·실행 생명주기에 반영한다.**

여기서 헷갈리기 쉬운 점은 **데이터 소스**와 **컨텍스트 타입**이 서로 다른 축이라는 거예요.

- **데이터 소스**: 판단에 필요한 값을 어디서 가져오는가? (`Runtime Context`, `State`, `Store`)
- **컨텍스트 타입**: 그 값을 이용해 무엇을 바꾸는가? (`Model Context`, `Tool Context`, `Life-cycle Context`)

```mermaid
flowchart LR
    subgraph Sources["① 데이터 소스: 어디서 값을 읽나"]
        RC["Runtime Context<br/>호출 때 주입되는 설정<br/>user_id, role, env"]
        ST["State<br/>현재 대화의 단기 상태<br/>messages, 인증, 업로드 파일"]
        STORE["Store<br/>대화 밖 장기 메모리<br/>선호도, 히스토리"]
    end

    subgraph Mechanisms["② 구현 위치: 값을 읽어 결정하는 코드"]
        DP["@dynamic_prompt<br/>시스템 프롬프트 생성"]
        WMC["@wrap_model_call<br/>메시지·도구·모델 수정"]
        TR["ToolRuntime<br/>도구 내부에서 읽기·쓰기"]
        MW["Middleware hooks<br/>요약·가드레일·로깅"]
    end

    subgraph Effects["③ 컨텍스트 타입: 실제로 무엇이 바뀌나"]
        MC["Model Context<br/>이번 모델 호출에 보이는 내용"]
        TC["Tool Context<br/>도구가 읽고 저장하는 상태"]
        LC["Life-cycle Context<br/>에이전트 실행 흐름과 기록"]
    end

    RC --> DP
    ST --> DP
    STORE --> DP
    RC --> WMC
    ST --> WMC
    STORE --> WMC
    RC --> TR
    ST --> TR
    STORE --> TR
    ST --> MW

    DP --> MC
    WMC --> MC
    TR --> TC
    MW --> LC

    classDef source fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef mechanism fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef effect fill:#cce5ff,stroke:#007bff,color:#004085

    class RC,ST,STORE source
    class DP,WMC,TR,MW mechanism
    class MC,TC,LC effect
```

### 차트 읽는 법

1. **왼쪽(데이터 소스)**은 "판단 근거"예요. 예를 들어 `user_role`, `messages`, 사용자 선호도 같은 값입니다.
2. **가운데(구현 위치)**는 그 값을 실제로 읽는 코드예요. 이 장에서는 `@dynamic_prompt`, `@wrap_model_call`, `ToolRuntime`, `SummarizationMiddleware`를 사용합니다.
3. **오른쪽(컨텍스트 타입)**은 최종적으로 바뀌는 대상이에요. 모델에게 보이는 내용이 바뀌면 Model Context, 도구가 State/Store를 읽거나 쓰면 Tool Context, 실행 과정 자체가 요약·로깅·가드레일로 바뀌면 Life-cycle Context입니다.

> 예: `Runtime Context → @dynamic_prompt → Model Context`는 "호출 시 전달된 사용자 역할을 읽어서 이번 모델 호출의 시스템 프롬프트를 다르게 만든다"는 뜻이에요.
>
> 예: `Store → ToolRuntime → Tool Context`는 "도구가 장기 저장소에서 사용자 설정을 읽거나 새 설정을 저장한다"는 뜻이에요.

> 🔑 **핵심 개념**: Model Context 변경은 보통 **단일 모델 호출에만 적용되는 일시적(Transient)** 변경이에요. 반대로 Tool Context와 Life-cycle Context는 State/Store를 수정해 이후 턴에도 영향을 줄 수 있으므로 **지속적(Persistent)** 변경으로 봅니다.


## 컨텍스트 타입과 데이터 소스 정리

컨텍스트 엔지니어링을 설계할 때는 먼저 **변경 대상**을 고르고, 그 다음 **데이터 소스**를 고르면 이해가 쉬워요.

### 1) 무엇을 바꾸는가? — 컨텍스트 타입

| 구분 | 쉽게 말하면 | 바꾸는 대상 | 지속성 | 구현 방법 |
|------|------------|------------|--------|----------|
| **Model Context** | "이번 모델 호출에 무엇을 보여줄까?" | 시스템 프롬프트, 메시지, 도구 목록, 모델, 응답 형식 | 주로 Transient | `@dynamic_prompt`, `@wrap_model_call` |
| **Tool Context** | "도구가 무엇을 읽고 저장할 수 있을까?" | `runtime.state`, `runtime.context`, `runtime.store`, `Command(update=...)` | Persistent 가능 | `ToolRuntime`, `Command(update=...)`, `store.put()` |
| **Life-cycle Context** | "에이전트 실행 중간에 어떤 공통 처리를 할까?" | 요약, 가드레일, 로깅, 실행 제어 | Persistent 가능 | `SummarizationMiddleware` 등 미들웨어 훅 |

### 2) 어디서 값을 가져오는가? — 데이터 소스

| 데이터 소스 | 다른 이름 | 범위 | 예시 | 선택 기준 |
|------------|---------|------|------|----------|
| **Runtime Context** | 호출 시 주입되는 정적 구성 | 호출 단위 | `user_id`, `user_role`, API 키, 배포 환경 | 서버가 검증해 넣어주는 값, 사용자 요청문에 직접 넣으면 위험한 값 |
| **State** | 단기 메모리 | 대화 세션 | 메시지, 인증 상태, 업로드 파일, 현재 작업 단계 | 현재 대화 안에서 변하고 다음 턴에도 이어져야 하는 값 |
| **Store** | 장기 메모리 | 대화 간 공유 | 사용자 선호도, 검색 히스토리, 장기 기억 | 새로운 대화에서도 유지되어야 하는 사용자/업무 데이터 |

### 3) 3×3 매트릭스로 보는 대표 설계 질문

아래 표는 "데이터 소스 × 컨텍스트 타입" 조합을 설계 질문으로 바꾼 것입니다. 모든 칸을 매번 구현해야 한다는 뜻이 아니라, **필요한 칸만 선택하는 체크리스트**로 보면 됩니다.

| 데이터 소스 ↓ / 변경 대상 → | **Model Context**<br/>모델 호출 제어 | **Tool Context**<br/>도구 읽기·쓰기 | **Life-cycle Context**<br/>실행 흐름 제어 |
|---|---|---|---|
| **Runtime Context** | 역할/환경에 따라 프롬프트·도구·모델을 다르게 할까? | 도구가 `user_id`, 권한, 연결 정보를 안전하게 사용해야 할까? | 운영 환경이나 플래그에 따라 로깅·가드레일 정책을 바꿀까? |
| **State** | 메시지 수, 인증 상태, 업로드 파일을 모델 요청에 반영할까? | 도구가 현재 세션 상태를 읽거나 `Command(update=...)`로 갱신해야 할까? | 메시지 수/토큰 수가 많아지면 요약해야 할까? |
| **Store** | 사용자 선호도나 장기 기억으로 응답 스타일을 개인화할까? | 도구가 설정, 히스토리, 메모리를 읽고 저장해야 할까? | 저장된 정책이나 장기 기억을 바탕으로 공통 처리를 조정할까? |

### 4) 이 노트북의 예제가 차트에서 차지하는 위치

| 섹션 | 차트의 흐름 | 배우는 API | 핵심 질문 |
|---|---|---|---|
| 1장 | State/Store/Runtime Context → `@dynamic_prompt` → Model Context | `@dynamic_prompt` | 시스템 프롬프트를 상황별로 어떻게 바꿀까? |
| 2장 | State → `@wrap_model_call` → Model Context | `request.override(messages=...)` | 모델에게 추가 파일 정보를 어떻게 주입할까? |
| 3장 | State/Runtime Context → `@wrap_model_call` → Model Context | `request.override(tools=...)` | 사용자가 볼 수 있는 도구 목록을 어떻게 제한할까? |
| 4장 | Runtime Context/State/Store → `ToolRuntime` → Tool Context | `ToolRuntime`, `Command`, `store.put()` | 도구가 상태를 읽고 저장하려면 무엇이 필요할까? |
| 5장 | State → Middleware hooks → Life-cycle Context | `SummarizationMiddleware` | 대화가 길어질 때 실행 과정에서 자동으로 무엇을 할까? |
| 6장 | 여러 흐름 조합 | 미들웨어 리스트 조합 | 실무 에이전트에서 여러 컨텍스트 결정을 어떻게 함께 적용할까? |


## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키 등을 읽어요)
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정 (에이전트 컨텍스트 흐름을 모니터링해요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Context-Engineering"

# LangSmith 추적이 활성화되었어요.
# print(f"프로젝트: {os.environ['LANGCHAIN_PROJECT']}")

## 1. Model Context: 동적 시스템 프롬프트 (`@dynamic_prompt`)

Model Context는 **LLM이 이번 호출에서 실제로 보게 되는 입력**을 뜻해요. 시스템 프롬프트, 대화 메시지, 사용 가능한 도구, 모델 종류, 응답 형식이 모두 Model Context에 포함됩니다.

`@dynamic_prompt`는 그중에서도 **시스템 프롬프트를 매번 다시 만드는 가장 단순한 미들웨어**예요. 고정된 프롬프트 대신 State/Store/Runtime Context를 읽어서 사용자별·상황별 지시사항을 만듭니다.

`@dynamic_prompt` 함수는 `ModelRequest`를 받아 문자열을 반환해요. 반환된 문자열이 이번 모델 호출의 시스템 프롬프트로 사용됩니다.

이 절을 읽을 때는 각 예제를 다음 네 단계로 따라가면 됩니다.

1. **어떤 데이터 소스를 읽는가?** — `request.messages`, `request.runtime.store`, `request.runtime.context`
2. **무슨 판단을 하는가?** — 메시지가 긴가, 선호도가 있는가, 역할이 무엇인가
3. **프롬프트가 어떻게 바뀌는가?** — 간결하게 답하기, 친근한 톤, 읽기 전용 권한 안내
4. **지속되는가?** — 프롬프트 변경 자체는 이번 모델 호출에만 적용되므로 보통 Transient입니다.

> 💡 **핵심 정리**: `@dynamic_prompt`는 **State 기반**, **Store 기반**, **Runtime Context 기반** 세 가지 방식으로 활용할 수 있어요. 같은 "프롬프트 변경"이라도 어떤 데이터 소스를 읽는지에 따라 설계 의도가 달라집니다.


### 1-1. State 기반 동적 프롬프트

대화 State에 따라 시스템 프롬프트를 조정해요. 예를 들어, 메시지가 많아질수록 더 간결하게 응답하도록 지시할 수 있어요.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 생성 및 미들웨어 관련 import
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-2. Store 기반 동적 프롬프트

Store(장기 메모리)에 저장된 사용자 선호도를 읽어 개인화된 시스템 프롬프트를 만들어요. 같은 질문에 사용자마다 다른 스타일로 답변하는 개인화 경험을 구현할 수 있어요.

> 🔑 **핵심 개념**: `request.runtime.store`로 Store에 접근해요. `store.get((네임스페이스,), 키)` 형태로 데이터를 읽어요.

In [ ]:
from dataclasses import dataclass
from langgraph.store.memory import InMemoryStore
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 생성 (Store와 Context 스키마 연결)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: bob: Store에 선호도 없음 → 기본 스타일
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-3. Runtime Context 기반 동적 프롬프트

에이전트 호출 시 전달하는 Runtime Context를 기반으로 시스템 프롬프트를 동적으로 생성해요. Runtime Context 값은 한 번의 실행 중에는 변하지 않는 정적 구성값이지만, 호출마다 다른 `user_role`, `deployment_env` 등을 주입할 수 있으므로 결과 프롬프트는 달라집니다.

> 💡 **실무 팁**: Runtime Context는 인증 레이어에서 검증된 정보(user_role, permissions)를 전달하기에 적합해요. State나 Store와 달리 에이전트 외부에서 주입되므로 변조 위험이 낮아요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 역할과 환경을 포함한 Runtime Context 스키마
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: viewer 사용자 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. Model Context: 메시지 주입 (`@wrap_model_call`)

`@dynamic_prompt`가 시스템 프롬프트 문자열만 바꾼다면, `@wrap_model_call`은 모델 호출 전체를 감싸서 **메시지, 도구 목록, 모델, 응답 형식**까지 수정할 수 있어요.

실행 흐름은 다음과 같습니다.

1. 모델 호출 직전에 `request`를 받습니다.
2. `request.state`, `request.runtime.context`, `request.runtime.store` 등을 읽습니다.
3. 필요하면 `request.override(...)`로 Model Context를 바꿉니다.
4. 마지막에 `handler(request)`를 호출해 실제 모델 호출을 진행합니다.

이 절의 예제는 **State에 들어 있는 업로드 파일의 실제 텍스트 내용**을 읽어 메시지로 주입합니다. 즉, 차트로 보면 `State → @wrap_model_call → Model Context` 흐름이에요.

> 💡 **핵심 정리**: `request.override(messages=..., tools=..., model=...)`는 "이번 모델 호출에 전달되는 요청"을 바꾸는 작업입니다. State/Store를 저장하는 작업과는 구분해서 보세요.


### 2-1. State에서 파일 컨텍스트 주입

사용자가 업로드한 파일 정보를 State에서 읽어 모델 요청에 자동으로 추가해요. 파일 처리 에이전트를 만들 때 유용한 패턴이에요.

아래 실습은 단순히 `summary`만 넣는 가짜 업로드가 아니라, 노트북 실행 중 **샘플 CSV/Markdown 파일을 직접 생성**하고 `Path.read_text()`로 파일 내용을 읽은 뒤 `uploaded_files` State에 넣습니다. 그래서 모델은 "파일을 직접 열 수 없다"고 답하는 대신, 미들웨어가 주입한 실제 파일 내용에 근거해서 계산·요약할 수 있어요.


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelResponse
from pathlib import Path
from textwrap import dedent
from typing import Callable

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: wrap_model_call 및 타입 힌트 import
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Model Context: 도구 필터링

모든 사용자에게 모든 도구를 제공하는 것은 보안과 UX 모두에서 좋지 않아요. **State 기반** 또는 **Runtime Context 기반**으로 도구를 동적으로 필터링할 수 있어요.

> **왜 도구 필터링이 필요할까요?** 음식점에서 어린이 메뉴와 성인 메뉴를 분리하는 것과 같아요. 어린이에게 전체 메뉴판(50가지 도구)을 주면 혼란스럽고, 주류 메뉴(위험한 도구)에 접근할 수도 있어요. 사용자 권한에 맞는 도구만 보여주면 LLM이 더 정확한 선택을 하고, 보안 사고도 방지할 수 있어요.

### 도구 필터링 접근 방식 비교

| 기준 | 데이터 소스 | 적합한 상황 |
|------|-----------|------------|
| **인증 상태** | State (`authenticated`) | 로그인 전/후 도구 차별화 |
| **사용자 역할** | Runtime Context (`user_role`) | RBAC 기반 접근 제어 |
| **구독 등급** | Runtime Context (`subscription`) | SaaS 기능 제한 |
| **대화 진행도** | State (`len(messages)`) | 점진적 기능 해제 |

> ⚠️ **자주 하는 실수**: 도구 필터링은 LLM이 도구를 **보지 못하도록** 막는 것이에요. 도구가 존재하지 않으면 LLM은 호출 자체를 시도하지 않아요. `request.override(tools=filtered_tools)`로 구현해요.

### 3-1. State 기반 도구 필터링 (인증 상태)

State의 `authenticated` 필드에 따라 공개 도구와 비공개 도구를 구분해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 공개/비공개 도구 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 인증된 사용자 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. Runtime Context 기반 RBAC (역할 기반 접근 제어)

역할 기반 접근 제어(RBAC, Role-Based Access Control)를 미들웨어로 구현해요. 사용자 역할과 구독 등급에 따라 사용 가능한 도구를 제한합니다.

> 🔑 **핵심 개념**: RBAC 미들웨어는 **인증 레이어에서 검증된 역할 정보**를 Runtime Context로 받아요. 사용자가 직접 수정할 수 없는 서버 사이드 정보여서 보안상 안전해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 데이터 CRUD 도구 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: admin: 동일한 삭제 요청
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. Tool Context: 도구에서 데이터 소스 읽기/쓰기

Tool Context는 **도구 함수가 실행되는 동안 사용할 수 있는 컨텍스트**예요. LLM은 도구의 일반 인자(`query`, `password` 등)만 채우고, `ToolRuntime`은 LangChain 런타임이 자동으로 주입합니다.

도구 안에서는 `ToolRuntime`을 통해 세 가지 데이터 소스에 접근할 수 있어요.

- `runtime.context`: 호출 시 서버가 넣어준 Runtime Context (`user_id`, 권한, 환경 등)
- `runtime.state`: 현재 대화의 State (`messages`, 인증 상태, 세션 정보 등)
- `runtime.store`: 대화 밖에도 유지되는 장기 Store (선호도, 히스토리 등)

Model Context와의 차이는 **쓰기(write)** 가능성입니다. Model Context는 주로 "이번 호출에 무엇을 보여줄까"에 가깝지만, Tool Context는 `Command(update=...)`나 `store.put()`을 통해 이후 턴에도 남는 값을 만들 수 있어요.

> 💡 **핵심 정리**: `ToolRuntime`은 도구 함수의 **특별한 매개변수**예요. 에이전트가 도구를 호출할 때 자동으로 주입되므로, LLM이 이 인자를 채울 필요가 없어요. `ToolRuntime[ContextType]` 타입 힌트로 Runtime Context 타입도 지정할 수 있습니다.


### 4-1. State에서 읽기 (`runtime.state`)

In [ ]:
from langchain.tools import ToolRuntime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ToolRuntime import (도구에서 State/Store에 접근해요)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-2. Store에서 읽기 (`runtime.store`)

Store는 대화 세션을 넘어 지속되는 장기 데이터를 저장해요. `ToolRuntime[ContextType]` 타입 힌트로 Runtime Context 타입을 함께 지정할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Store 읽기용 Context 스키마
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-3. State에 쓰기 (`Command(update=...)`)

도구에서 `Command(update={...})`를 반환하면 State를 직접 업데이트할 수 있어요. 인증 도구가 인증 상태를 State에 기록하거나, 세션 데이터를 저장하는 패턴에 사용해요.

> ⚠️ **자주 하는 실수**: `Command`를 반환하는 도구는 반드시 `ToolMessage`도 함께 반환해야 해요. `tool_call_id`가 없으면 에러가 발생해요.

In [ ]:
from langgraph.types import Command
from langchain.messages import ToolMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Command와 ToolMessage import
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-4. Store에 쓰기 (`runtime.store.put()`)

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: @tool
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Life-cycle Context: SummarizationMiddleware

Life-cycle Context는 "모델에게 무엇을 보여줄까"나 "도구가 무엇을 저장할까"보다 한 단계 바깥에 있는 개념이에요. 에이전트가 실행되는 동안 **모델 호출 전후, 도구 호출 전후, 전체 실행 시작/종료 시점**에 공통 처리를 넣는 영역입니다.

대표적인 예시는 다음과 같아요.

- 대화가 길어지면 오래된 메시지를 요약하기
- 위험한 요청을 가드레일로 차단하기
- 도구 호출 결과와 비용을 로깅하기
- 특정 조건에서 실행 흐름을 건너뛰거나 중단하기

여기서는 가장 이해하기 쉬운 예로 `SummarizationMiddleware`를 사용합니다. 대화 히스토리가 너무 길어지면 오래된 메시지를 요약으로 대체해 **State를 영구적으로 수정**해요. 따라서 이후 턴은 원본 긴 메시지 대신 요약된 맥락을 보게 됩니다.

> 💡 **실무 팁**: `SummarizationMiddleware`는 `trigger=("tokens", N)`으로 토큰 수 기준, `trigger=("messages", N)`으로 메시지 수 기준으로 요약을 트리거할 수 있어요. `keep=("messages", N)`으로 요약 후 보존할 최근 메시지 수를 지정해요.


In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: SummarizationMiddleware import
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 종합 예제: 멀티 레이어 컨텍스트 엔지니어링

지금까지는 컨텍스트 결정을 하나씩 분리해서 봤어요. 실무에서는 보통 여러 결정을 동시에 적용합니다. 예를 들어 구독 기반 퍼스널 어시스턴트는 다음 질문을 한 번에 해결해야 해요.

1. **이 사용자는 누구인가?** → Runtime Context의 `user_id`, `user_role`, `subscription`
2. **이 사용자에게 어떤 말투와 안내를 줄 것인가?** → `@dynamic_prompt`로 Model Context 변경
3. **이 사용자가 어떤 도구를 볼 수 있는가?** → `@wrap_model_call`로 도구 목록 필터링
4. **검색 히스토리를 어디에 저장하고 다시 읽을 것인가?** → `ToolRuntime`과 Store로 Tool Context 처리

이 예제는 다음을 동시에 적용해요.

- **Runtime Context** → 사용자 역할 + 구독 등급
- **Store** → 사용자 검색 히스토리 (장기 메모리)
- **`@dynamic_prompt`** → 역할/등급 기반 시스템 프롬프트
- **`@wrap_model_call`** → 구독 등급 기반 도구 필터링
- **`ToolRuntime`** → 도구에서 Store 읽기/쓰기

> 💡 **핵심 정리**: 미들웨어는 `middleware=[m1, m2, m3]` 리스트로 조합해요. 순서대로 적용되므로 `@dynamic_prompt` → `@wrap_model_call` 순으로 나열하면 프롬프트 설정 후 도구 필터링이 이루어져요. 이때 각 미들웨어가 "어떤 데이터 소스를 읽고 어떤 컨텍스트 타입을 바꾸는지"를 주석으로 남겨두면 유지보수가 쉬워집니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 생성 (여러 미들웨어 조합)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: === 무료 사용자 ===
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## TODO: 실습 — 사용자 등급별 응답 차별화

아래 에이전트를 완성하여 `gold` / `silver` / `bronze` 등급에 따라 다른 도구 접근 권한과 시스템 프롬프트를 적용해보세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 아래 코드를 완성해보세요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **컨텍스트 3종류**: Model Context(Transient - 단일 호출), Tool Context(Persistent - State/Store 쓰기), Life-cycle Context(Persistent - 요약/가드레일)
- **데이터 소스 3종류**: Runtime Context(호출 시 주입되는 정적 구성값), State(대화 세션 단기 메모리), Store(대화 간 장기 메모리)
- **`@dynamic_prompt`**: State, Store, Runtime Context를 읽어 시스템 프롬프트를 동적으로 생성해요
- **`@wrap_model_call`**: 메시지 주입, 도구 필터링, 모델 선택, 응답 형식을 런타임에 제어해요
- **RBAC 구현**: `@wrap_model_call` + Runtime Context로 역할 기반 도구 접근 제어를 구현해요
- **`ToolRuntime`**: 도구 함수에서 State/Store/Context에 접근하는 특별 매개변수 (LLM이 채우지 않아요)
- **`Command(update=...)`**: 도구에서 State를 영구적으로 업데이트하는 방법
- **`SummarizationMiddleware`**: 토큰 초과 시 자동 요약으로 대화 히스토리를 압축해요

## 다음 노트북 예고

다음 `04-Prebuilt-Middleware.ipynb`에서는 **LangChain V1이 기본 제공하는 12+ 내장 미들웨어**를 배워요. Summarization, PII 보호, 호출 횟수 제한, Fallback, Tool Retry, LLM-as-ToolSelector, TodoList, Context Editing 등 자주 쓰는 패턴이 이미 구현돼 있어요. 이번 노트북에서 익힌 컨텍스트 3요소가 각 내장 미들웨어에 어떻게 녹아 있는지 확인해볼게요.